**Creating the headers for the CSV file**

In [1]:
import csv
data = [
    ["Date",
     "Team_1_id", "Team_1_name", "Team_1_score", "Team_1_spread", 
     "Team_2_id", "Team_2_name", "Team_2_score", "Team_2_spread",
     "Predicted_winner_id", "Predicted_winner_name"],
    # Add your data rows here
]
count = 1

**Importing in the URL**

In [2]:
from bs4 import BeautifulSoup
import requests

def process_url(url):
    page = requests.get(url)
    
    soup = BeautifulSoup(page.text, 'html.parser')
    
    #print(soup)
    return soup

**Extracting Team IDs from all the teams before results are announced**

In [3]:
def ids_before_results(soup):
    logos = soup.find_all("div", class_="TeamLogo TeamLogo--small")
    
    team_ids = [logo.find("img")["data-lazy"][-9:-4] for logo in logos]
    
    #print(team_ids)
    return team_ids

**Extracting Team IDs and Predicitions from the teams predicted to win by the EXPERTS**

In [4]:
def ids_expert_picks(soup):
    spread_divs = soup.find_all("div", class_="expert-spread")
    
    predicted_team_ids = [logo.find("img")["data-lazy"][-9:-4] for logo in spread_divs]
    
    # print(predicted_team_ids)
    # print()
    return predicted_team_ids

def expert_predictions(soup):
    spread_divs = soup.find_all("div", class_="expert-spread")
    predictions = [div.get_text(strip=True).split(maxsplit=1) for div in spread_divs]
    
    # print(predictions)
    return predictions

**Extracting all the team names**

In [5]:
def extract_names(soup):
    sports_teams = soup.find_all("span", attrs={"class": "team"})
    
    team_names = [team.get_text(strip=True) for team in sports_teams]
    
    # print(team_names)
    
    return team_names

**Extracting Game odds before the game**

In [6]:
def sportsbook_predicitions(soup):
    game_odds = soup.find_all("div", class_ = "game-odds")
    predictions = [div.get_text(strip=True) for div in game_odds]

    odds_lst = []
    for i in range(1, len(predictions)):
        if i%2 != 0:
            odds_lst.append(predictions[i])
    
    return odds_lst

**Extracting scores all the Team**

In [7]:
def extract_scores(soup):
    teams_score = soup.find_all("span", attrs={"class": "score"})
    
    teams_scores_strip = [score.get_text(strip=True) for score in teams_score]
    
    # print(teams_scores_strip)
    
    return teams_scores_strip

**Kind of Irrelevent Cell but it just matches teams with their scores**

In [8]:
def sample_match_teams_score(names, scores):
    count = 1
    for name, score in zip(names, scores):
        # print(count, ". ", name, "-", score)
        count += 1

**Putting all the data together and also appending the data into the csv file**

In [9]:
def final_match_teams_score(date, names, scores, ids, predictions, predicted_team_ids, odds_lst):
    # games = [(names[i], names[i+1], scores[i], scores[i+1], ids[i], ids[i+1]) for i in range(0, len(team_names), 2)]
    global count
    
    games = []
    for i in range(0, len(team_names), 2):
        team1_name = names[i]
        team2_name = names[i+1]
        team1_score = scores[i]
        team2_score = scores[i+1]
        team1_id = ids[i]
        team2_id = ids[i+1]
        games.append((team1_name, team2_name, team1_score, team2_score, team1_id, team2_id))
    
    # matched_games = [(games[i], predictions[i], predicted_team_ids[i], odds_lst[i]) for i in range(len(predictions))]
    
    matched_games = []
    for i in range(len(predictions)):
        game = games[i]
        prediction = predictions[i]
        predicted_team_id = predicted_team_ids[i]
        odds = odds_lst[i]
        
        
        if "-" in odds:
            data[count].insert(8, odds)
        else:
            data[count].insert(4, odds.replace("+", "-"))
            
        if "+" in odds:
            data[count].insert(8, odds)
        else:
            data[count].insert(4, odds.replace("-", "+"))
        
        data[count].insert(0, date)
        
        data[count].insert(1, game[4])
        data[count].insert(2, game[0])
        data[count].insert(3, game[2])
        
        data[count].insert(5, game[5])
        data[count].insert(6, game[1])
        data[count].insert(7, game[3])
        
        data[count].insert(9, predicted_team_id)
        data[count].insert(10, prediction[0])
        count += 1
        data.append([])
        
        matched_games.append((game, prediction, predicted_team_id, odds))
        
    
    # for game, outcome_name, outcome_id in matched_games:
    #     print(f"[{game[4]}] {game[0]} {game[2]} vs [{game[5]}] {game[1]} {game[3]} - Outcome: [{outcome_id}] {outcome_name[0]}\n")

    return matched_games

**Display the data for the month**

In [10]:
from datetime import datetime, timedelta

start_date = datetime(2024, 11, 4)
end_date = datetime.now() - timedelta(days=1)

current_date = start_date
while current_date <= end_date:
    i = current_date.strftime("%Y%m%d")
    current_date += timedelta(days=1)

    if count >= len(data):
        data.append([])

    url = f'https://www.cbssports.com/college-basketball/expert-picks/{i}/'
    soup = process_url(url)

    team_ids = ids_before_results(soup)
    predicted_team_ids = ids_expert_picks(soup)
    predictions = expert_predictions(soup)
    team_names = extract_names(soup)
    team_scores = extract_scores(soup)
    odds_lst = sportsbook_predicitions(soup)
    sample_match_teams_score(team_names, team_scores)
    games = final_match_teams_score(i, team_names, team_scores, team_ids, predictions, predicted_team_ids, odds_lst)

**Write Data into a CSV file**

In [11]:
with open('scraped_ncaa_scores.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(data)